# Industry GNN V4 - Point-in-Time, Conservative Backtest and HIST-Factor

这是新的独立执行入口，不依赖先前 Notebook 的变量状态。它完成四件事：

1. 对每类特征、指数成分、行业归属和图输入生成逐项 Point-in-Time 证据表，保留 `PARTIAL/UNVERIFIED`，不伪造 `VERIFIED`；
2. 开盘成交只使用开盘价、当日涨跌停价、停牌/缺失报价，并在无法确认排队成交时采用“不成交”的保守假设；
3. 用同一 Top-10、同一历史税费和同一持有区间比较 Ridge、Shared MLP、Industry MLP、Residual-GCN、Residual-GAT Complete、Residual-GAT Sparse；
4. 增加 `HIST-Factor`：实现论文图中的预定义概念、隐藏概念、个股信息和双残差主结构。

`HIST-Factor` 是论文机制在本项目 12 个周频因子上的适配组，不冒充 Alpha360 + 两层 GRU + 行业/主营业务概念的逐字复现。Smoke 只验证流程；正式结论只能来自 Full Run 的真实输出。


## 1. 模式、项目目录与 Full Run 安全门


In [ ]:
import sys

RUN_MODE = "SMOKE_TEST"
CONFIRM_FULL_RUN = False
INSTALL_OR_VERIFY_DEPENDENCIES = True
PROJECT_DIRECTORY_NAME = "Industry_GNN_Research_PIT_HIST_V4"

assert RUN_MODE in {"SMOKE_TEST", "FULL_RUN"}
if RUN_MODE == "FULL_RUN" and not CONFIRM_FULL_RUN:
    raise RuntimeError("FULL_RUN is gated. First complete SMOKE_TEST, then set CONFIRM_FULL_RUN=True.")
if sys.version_info[:2] != (3, 11):
    raise RuntimeError(f"请选择 Python 3.11 Miniconda Kernel；当前是 {sys.version.split()[0]}")
print("Run mode:", RUN_MODE)


## 2. 安装或核对依赖

始终使用当前 Notebook Kernel。已满足版本时不会重复下载；FULL_RUN 若无 CUDA 会立即停止。


In [ ]:
import json, subprocess, sys

def install(args):
    print(">", " ".join(map(str, args)), flush=True)
    subprocess.check_call(list(map(str, args)))

if INSTALL_OR_VERIFY_DEPENDENCIES:
    probe = subprocess.run(
        [sys.executable, "-c", "import json,torch;print(json.dumps({'cuda':torch.version.cuda}))"],
        capture_output=True, text=True,
    )
    cuda_build = None
    if probe.returncode == 0:
        try: cuda_build = json.loads(probe.stdout.strip().splitlines()[-1])["cuda"]
        except Exception: pass
    if cuda_build is None:
        install([sys.executable, "-m", "pip", "install", "--upgrade", "--force-reinstall",
                 "torch==2.12.1", "--index-url", "https://download.pytorch.org/whl/cu126"])
    install([sys.executable, "-m", "pip", "install", "--upgrade",
             "torch-geometric==2.8.0", "tushare==1.4.24", "pandas==2.3.3", "numpy==2.2.6",
             "pyarrow==21.0.0", "scikit-learn==1.7.2", "scipy==1.16.3", "networkx==3.5",
             "PyYAML==6.0.3", "matplotlib==3.10.7", "seaborn==0.13.2", "tqdm==4.67.1",
             "psutil==7.1.2", "joblib==1.5.2", "pytest==8.4.2", "nbformat==5.10.4",
             "nbclient==0.10.2", "ipykernel==6.30.1"])
    install([sys.executable, "-m", "pip", "check"])


## 3. 恢复内嵌源码并立即定义 CONFIG

此单元保留已有缓存/检查点，只更新版本化源码与配置。执行后 `PROJECT_ROOT` 和 `CONFIG` 已定义，不再依赖隐藏的前置单元。


In [ ]:
import base64, io, sys, zipfile
from pathlib import Path

WORKSPACE_ROOT = Path.cwd().resolve()
if (WORKSPACE_ROOT / "config.yaml").exists() and (WORKSPACE_ROOT / "src" / "industry_gnn").exists():
    PROJECT_ROOT = WORKSPACE_ROOT
else:
    PROJECT_ROOT = WORKSPACE_ROOT / PROJECT_DIRECTORY_NAME
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

PROJECT_ARCHIVE_B64 = "..."
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PROJECT_ARCHIVE_B64))) as archive:
    archive.extractall(PROJECT_ROOT)

NOTEBOOK_B64 = "..."
if NOTEBOOK_B64:
    (PROJECT_ROOT / "Industry_GNN_Research_PIT_HIST_V4_From_Zero.ipynb").write_bytes(base64.b64decode(NOTEBOOK_B64))

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from industry_gnn.pipeline import initialize
CONFIG = initialize(PROJECT_ROOT, PROJECT_ROOT / "config.yaml", RUN_MODE)
print("Project:", PROJECT_ROOT)
print("CONFIG defined:", CONFIG["formal_protocol"]["protocol_id"])
print("Splits:", CONFIG["splits"])


## 4. GPU、CUDA 与图算子实测


In [ ]:
import torch, torch_geometric
from torch_geometric.nn import GATConv, GCNConv

print("PyTorch:", torch.__version__, "CUDA runtime:", torch.version.cuda)
print("PyG:", torch_geometric.__version__, "CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("未检测到 CUDA GPU；不会在 CPU 上启动长时间实验。")
device = torch.device("cuda")
props = torch.cuda.get_device_properties(0)
print("GPU:", props.name, "VRAM GB:", round(props.total_memory / 2**30, 2))
x = torch.randn(6, 8, device=device, requires_grad=True)
edge = torch.tensor([[0,1,1,2,3,4,4,5],[1,0,2,1,4,3,5,4]], device=device)
loss = GCNConv(8,8).to(device)(x,edge).square().mean()
loss += GATConv(8,8,heads=2,concat=False).to(device)(x,edge).square().mean()
loss.backward(); torch.cuda.synchronize()
print("CUDA GCN/GAT forward/backward: PASS")


## 5. 下载或复用 TuShare 数据

只有新数据键缺失时才请求隐藏 Token。Token 不写入 Notebook、日志或产物。下载中断后重跑会复用已完整写入的分片。


In [ ]:
from industry_gnn.security import create_tushare_client, get_tushare_token
from industry_gnn.utils import config_hash
from industry_gnn.data import load_download_checkpoint
from industry_gnn.pipeline import data_stage

EXPECTED_DATA_KEY = config_hash({"project": CONFIG["project"], "data": CONFIG["data"]})
PRO = None
if load_download_checkpoint(PROJECT_ROOT, EXPECTED_DATA_KEY) is None:
    token_value = get_tushare_token()
    PRO = create_tushare_client(token_value)
    del token_value
REFERENCE, MARKET, DATA_KEY, DATA_CACHE_HIT = data_stage(PRO, CONFIG, PROJECT_ROOT)
print("Data cache hit:", DATA_CACHE_HIT, "key:", DATA_KEY)
print({name: len(frame) for name, frame in {**REFERENCE, **MARKET}.items()})


## 6. 构建因子、标签、历史行业图基础与交易表


In [ ]:
from IPython.display import display
from industry_gnn.pipeline import feature_stage

PANEL, EXECUTION_PRICES, BENCHMARK_PRICES, PANEL_META, PREPARED_KEY, PREPARED_CACHE_HIT = feature_stage(
    REFERENCE, MARKET, CONFIG, PROJECT_ROOT, DATA_KEY
)
display(PANEL.groupby("split").agg(
    decision_rows=("row_id","size"), labeled_rows=("label_available","sum"),
    dates=("signal_date","nunique"), stocks=("ts_code","nunique")))
assert PANEL["decision_eligible"].all()
assert PANEL[CONFIG["features"]["raw"]].notna().all().all()
print("Prepared cache hit:", PREPARED_CACHE_HIT, "key:", PREPARED_KEY)


## 7. 完整 Point-in-Time 审计

`PASS` 只表示代码时点和来源证据都能核验。若 TuShare 快照缺少历史公告/修订时间，即使计算顺序正确也保持 `PARTIAL` 或 `UNVERIFIED`。


In [ ]:
from industry_gnn.pit_audit import build_point_in_time_audit

PIT_AUDIT, PIT_SUMMARY = build_point_in_time_audit(PANEL, MARKET, REFERENCE, CONFIG, PROJECT_ROOT)
display(PIT_AUDIT[["component","field","code_timing_status","source_history_status","overall_status","limitation"]])
print(json.dumps(PIT_SUMMARY, ensure_ascii=False, indent=2))
assert not (PIT_AUDIT["overall_status"] == "FAIL").any()
if not PIT_SUMMARY["strict_point_in_time_claim_allowed"]:
    print("IMPORTANT: 本次结果只能标为 best-effort PIT / retrospective evidence，不能声称完全严格 PIT。")


## 8. 回测执行约束的自动测试

覆盖开盘涨跌停、盘中后来开板、停牌/无法卖出，以及基准首日对齐。


In [ ]:
subprocess.check_call([
    sys.executable, "-m", "pytest", str(PROJECT_ROOT / "tests" / "test_integrity.py"), "-q",
    "-k", "open_limit or backtest_carries_blocked_sale or benchmark_starts or historical_stamp_duty",
])
print("Conservative execution and benchmark alignment tests: PASS")


## 9. V3 残差模型与稀疏图设置

稀疏图仅使用信号日前交易日的 60 日收益相关性，在同一历史行业内选择 Top-5 邻居。


In [ ]:
import pandas as pd, numpy as np, json, hashlib

V3_ASSURANCE = "DIAGNOSTIC"
V3_PIT_LEDGER_CSV = PROJECT_ROOT / "pit_evidence" / "industry_membership_feature_ledger.csv"
V3_PIT_DAILY_PARQUET = PROJECT_ROOT / "pit_evidence" / "daily_returns_pit.parquet"
V3_PIT_PROVENANCE_JSON = PROJECT_ROOT / "pit_evidence" / "V3_PIT_PROVENANCE.json"
V3_K = 5
V3_CORR_LOOKBACK = 60
V3_HIDDEN_DIM = 32
V3_EPOCHS = int(CONFIG["models"]["max_epochs"])
V3_PATIENCE = int(CONFIG["models"]["patience"])
V3_TRAINING_SEEDS = list(CONFIG["models"]["training_seeds"])
V3_GRAPH_SEED = int(CONFIG["models"].get("random_graph_seeds", [1042])[0])
V3_OUTPUT_ROOT = PROJECT_ROOT / "V4_PIT_HIST_results" / RUN_MODE.lower()
V3_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Output root:", V3_OUTPUT_ROOT)


## 10. 载入 Compatible V3 的残差图实现


In [ ]:
import base64, zlib, hashlib
from pathlib import Path
_dst = PROJECT_ROOT / "V3_extension_modules" / "v3_residual.py"
_data = zlib.decompress(base64.b64decode("eNq9fWuT2zaW6Pf+FVhOZZd0U2xJfoyjDl3lijsb1zp2rt2Z7KxKxaJISKJbIjkk1d2Kb9/ffs8DIAGKajs7tZuacoskcHBwcN44wDiO87enopK1jKtkI+R9I/M6K/KZWMXZdpRsi1qm4te31yLep1lTizhPRZY3sior2cTLrRTrKi43An7GDXSsg7OzjxrcvpaiyLeHQFxvslrsinQPHd5f/e3qo0iLu3xbxGkt0riJCS5/SGTVZKtMwliiLvZVIv+tPku2cbYDTBr4t27iXVnPRLOROEIldvu6EbeyylYH+JOnRSXK/XKbJYTRRZznxT5P5E7mjQA8mqI6BGfvCwE/ks1oLYudbKosEaksobfME0D410OzKXLxNJiMz314usa24lU4DcbBmeM4Z38R330nfrx6924msl1ZVE19tqqKnYii1b7ZVzKK1HuB4zdMnDP1ahPXm2221I+f6yLXvyugRLFjWEgZmHldAzHU5/YVtyjjBgHpr7/CI39IivLQ9pGyxGc9RL7fwce4FnmpX5UwKryA/5UpA6iTrDwEQOumHbsuYVl3cV6pFjdbeM6DbZbD3whWV251049ZupYaONGZ+9BP3SjPz85+unp9/dvHq08iFHMHWCqaOL6gH8/1j+lY/3pBv24LZLVt1hzUpyVwIn07E8Z/2G6/k1GyifO1ZHDxDjihsV7BWuUFME9UxY3EF9tiHTWwYNtod9sHWS6xRZanwHHVIaokYnIrI4Xm4uzTr+/eXuNkvjhNFWe5MxOuMx1PXozGE/gf9p6Op5PRZDp6OnE8E7xzG2+zlPhE9ZpOrV5PB3ul8lZuixKZW3d7ZnV7PthtU2zTYt92sfF7obs8nP3y4c3Vu+j961/0EuHCYrN6E1cyjXbb0iKJegaNEq2TPEqKXbmVSFiLkIgBNYkbo0n3ri7jqrbesFgMfKjldhWhkgHqf7z6P7+9/Xj1JkKFhdiuZEyyGN+CNkNtFcWNha7WDan6sJO7pazqTVYefUplkqFqxMfF2dlZKleilvh5u3XxhzejOTKmAb7h15cgZ8HAW5KFAORpD7zWfSAg2Up9TvZpHGR1NwPXm5mfjO4dIgq7CLhJuvfQAfgT2BxkO2gKeot61L33hayqoqpDB3gVyNp2rOQ/9lkl3aTIQevDpH0BereO11JNEvADpSaWRbHtGuFACEf8Ld7u5RWCdh1cigtx9Z+/Xn18+8vV+2vx6frDrzPhiPMWJIxqaNPO2DAy9DsCBSW3Lv07w3m8AT34UxXvpC/UIteh1iW+UPjDPOJteF3tpSdGr6xuPA1Q5D8BXQXbOVioQtTZFu0EqKg9K21fAN3ldrSM0RTuc5B3YEGxyrZgBbN8LcDcIM5ZvpIVGA8JFhBhv9YrZhgtATIj8hhAiNd1Fl98QkW0iTOxLZKY0EhuuLlP9P3t+seAgL1RzDcT8QqGBSzXeazwhlW8l8m+oc81WrJmewCksV0BFg1xBAuC3QM9a/pbgowQRQO0Di5zXlJsa5b04i7KUhJ1Goz4hjRmDTKbsq6ctD+BaCBPuo28hzXjB1uDNnG1BmUp7xPAKGK+JEhI447L4dUTU5rhUS/zggC2LIpMnwDxYTYrWAn6iXPwgDGcXQbThuknaApy8Bm+1GB4SNIal1qN8FcZqAae9+B4NnzgBrf0Xo1xUruyOTDF+q1wscr5NxBtsQBpzmPXC+L8oP9FxaMwzaGRyMAJacC+9UcBNULwUSEAI/5jL1ER0iuR7kvyeGQ9iFvQfk/d+UnkOnza5sBRwJQjxXBlnFV6gJbcR/BO8oK9qItZyxvlPFkA17HGwgfQj9204TmAWTDZYL2ttcUuFygz4kuiV8+mGnZPm6D5Q4AT+r7IUWc40Jgdx2VfFlk+WSIRbp+esAzdBAWolcCYfoug0wolD1Mkyb4Sr3+6BhdXEZNoMgBc04tgd0OZoLPmGCq1HIBnGC4kQ16AD7fN/gBLEoYW7r2v3XgGhJZmMC9z1b867AZcefCfxeT5MVwGuiq22+JO7OLqRjas2i5FAV5+dYdGpQatPUrjg7j68AaikwSMYC3AA73ps6PJYvPZaGKwmYEfMtwPobDwNLgL2YNVrfF9pgIS5COSqgJCoVGWj4bYhFVCq7WGxX6fw7diewuW5X38HrAHq9MatCOIJQBZZXlGMtJBBptOLr2bNodShisIqxpj9fIiH3Gvr8Avg54WDtKqKAlroEDuztGS+uKneFvLhYJPgdtxx560tsLaV/N9tvl/R6DE/wXwQ1ZDj9ENQj2Zf1LRAVjClNX43zaeRefhscH6b2HwsWfTWZGZQHZsucxSUOgUFJBsx+nnCA1zpyrNN2q8zjQeIoi1U+OTbTwfjtkOueseRUGvNAkHv9HoIMYcqYoSXAZZXzD5oI3FJWILEbPTeqWmZzUgVw4LSFTiNNFxA8TZODOfHH8O5D9c8tAUHftxAvgMx530el4QL17CDIpbMJqcBMhoGZWkQmgfY8x8Kf42Ff/19lcyAR+uQVkW+/VmyFw4nCIA47eNmvi+yIvdIWLAvck81vAbpqWDEKE7k07pgG4PxnwuxaffMXJEV74qIBJPMPCEJnEJhhooolQS6LzBScFg8j4ywpvBGZ1s9W3TkfeC+6r8y4VcrRhRdmoJQ2zHeZamnSGgMYQ0a3l4vmUNPIjzqUbfgDJ31TmhC5BCWBBKE6FN+wZ0y7lTwpsiddB7AelOtvsUvuqPrFv2Nek6bEL82pqrG9+N/aWHs+HEQQDMvatdr5Osu6zZZKg7bEu9lM2dRM80Da51dOHGINPWi6XXURUUXZHMGZxvYn1zoskR7u7Q906VLywC/2sPWucELgIIYvtoel1AQt3KgDEU/2KSdWEGKj3/lpxz0sJq8SgOAI28BiUGfELgBkzeuir25fLgWj6sB8RGFyjYgdnz0G15hupS3gFrN+AnimfsFtfkBhXge6aG562j7QCDjegWg+H6Uacb869NRALkosnlkPXMjH9TIPNBqfzapad+HHwc9DIbpUBV6tAL8zDgCedmONdUcSpbBHlIToV11mgJIe4GbayFEiXXzCTLqSgttaI0irn0UIputVbunCSoZFm5dtRGvShwSz3NOhY3pHa0Y07rG2KdC3QyFS7HoY5NI3vOM5GacUzai2PSgTim9Yyc86TvnKSBCb7nnSNTpkGHzUnnXXGj4Q9JmI6y+sAYxyHD8biDzru9ZujBU0pIHg/FTcmjf8ynTSGAHmK6Uzy3MFzfvtOb5ZROFRaW/cGBT46GC9aNO5oE4/HkcWAYPIwm43Ew+e7bvSNzIPJnLIuWkkXDHN2JVt9g0jATlRRVWWA6ewT2DcMrAjjSjNBZOZKVItnjQ2vTUszD1xkbhyEKiRG8Hl4SU/2lPfVnC86f037KA9TOHOcBI9z88HlR1G/0A2Ue5+gtwguvzfJ95BWA4CDPVmB2xKefX0+fvxDAeWvcy4I4TqXcdrIitwoTfABqJy+SjUxulsU9J89+0RCI4zcx+Da4myN5awyCxW1cUd5QrLbxur6kPbUKw32JLmqOPqkKodFlAZOSiuWh3YKTlZ2jaxEOaZcooC0zF/d53P5kgZBxGjUwngvDFClMLXT2zeql43VaDGcV5WQvsBPpNNdhetabGGji+B11Pd9VEqm/ddT2jLg6zdaAY6i2tQJurLDsUFsewEaiatrIe+7hDnh9esYBhDpui64XhtyFgvP29QMai13cJJtLmpqQHB9w6I7CwERWvE1LiQEN2L0jX9fYM2xjl6G9ApSWumtxtLHRil/E4ldHlCcADo/QCzZh6/gsiuti9QjINpcEoADHQ6RZp0udLgG3NK4yAFbuIUxMh0ywRdvEw6ABhc2wyZpAqPaQfB2Tdd/s6AvMtI3vuUCsgs8wabD5x6BRUQ4ggsJHkt9T0rTHYLXXAaEWqAi6gvMFg6Lj1lRZCUyGphXlrJKwHqmo4ju1N3yhIkPd23ba9ED2nkQSgx+MsbGK2GjPW+1PYEoUE1Sgz6qKFpLdM9/Mj/lmGskXqPpA32yL4mYZJzfhi7EPRIXVxfRq+HTcKa4PuWFki2Utq1u2rIcWYre/QCoI6cjJf/JU21zjQe1L/AhoSt6sx+3z2og4Ve5tRPp/D89q4EuRS7BB3JyTEJzjqG1l1Xma6MMzGQwXBU2mmSqFIEG1Mf0NbGXm5JiNjVgbplKHOXvXruUCcU4cFn8+0qSdcfc0TAknqzkltXqAESloxNaJW9BiKTTugGcAVJndFo1LGiS0DJt2qzuvmu1fqGwqakK2dLolSoQawmJEHClAlnKJMTh8CVsm8doUVDDWtlLvpEYyBU3pKiZTTAuKQXEVRTx1+OWhNQqZn1FZh5DgTElUXe5RL+XhQMjBHiw08GYKVgAGHBCI99vGhU70zZ+jh11i6Oxm1szm7mc/8xFvGhx3PChaU7CUw8AfM/zALfDxc/cIuGT/En5ud2JpY/jUzFHiqsoXN+FzFrR4WZO8hoDGczXDYg9mYfG/SSJxTKPHKNKNYpNmZunfKs5vZKpnYvb4fKKHwjsLw88zjJmbLN/LoxYJJ5ZdJBvJElF5ni18/vF5sfCGwAKpUcND9GASfqbw1NN3uZn/2U88b2A+LO038hBu490yjcX9zB3dz8d69Pv5BMa3e8KCBlTRBOCR5RLmqQgHQVIw5PnsZmHxJ3RTTMWb9hBPMV9hHhYVMJi+KJXrSsKi0IchVlM8iPvxvoibBvMTdbSD9c8wYVeBzu+0/BswdAn6LWkBTogcIVRR38VlLX5/e/3z2/ed6VE4gP2Kk43aLswvAGfBKAV9JU85inYmqSCMaSMaTSNuc8NLLs7ZSFD0d2TR/q2GP7VRL2Xq+Qo8y66mQbF1BG+NCgYaZhzOgdOB4z0f/0CARitQ+40fkTBhI2bUZA8Y502IYT33ZTjLA4YGpihC77bveCDKMuVPLcq8XniqdM14FYbGY7MAjyGpiroetbQGo1o1GJzjWEaGknEaEOkOmKX/XEDZ8MFJrH1S4zgRBe0o9QeyQ5vPpO5/mA6IZk6jhbv43n06HuSyJx0Ez9IekRKANYTgfR2G/8X+MoQFDZJNkSVqF5zA+JgZC6d+JcttnMiQfEBb7uqJ30xCaj6PF5f11G+m6nFpayU1xS/QA1tBv+mD98OzE1oIeHPiwz/T0MUhphCdUC+vDxMbksvJPIUVGtjNeHViBPUVjPQO3F6Xhpl43uXxB5xTT93oNvuSsj/zFt2Fd6mJEeLLS00LfJqeqamxjgGBQekYVLW1VnYNqjpTFlR+TGGgfBWucktDpLDGrT7suNgxSrOVqldxDZEDsBcC+WlKnKPe+orE4CetgVdR9pcZhJh7kOl5o5SqIZQLH1Q99F83Gx7dcG9wpidBsFZoifEYHMIFTbYFqVZUib4dmdMw/gQ2Zj4rrqr4EMFzvHUVxXyeNeugoSaAhK+QoZwdaVal0bXKV1kF8JTbXI0V3D2m22y1207JGIvLeEkBkvPBKtAOjtp+EIQzgzKLaQeUYEQwFLrSwFYJ10+pcAgLRHsxD/lj3AstRzgZP5sqRaR6h+qvlUxqvWudy4aFaBtq370pyLUG4e0o0TZShUttIw4S4kNobUy0Yxs7LxnK43hBOqEZbm5WPtAGQpsfRBKEIe4US6etypsv/Ekw7jVpqyLbZoPeveGDtkC69CalK1VozXUvDvvJKoDV+UpKlFjJzcbYnSc/+WSUC0TzgRKsmfx2dfUPzT/xNhxw0Q23ibymm/CmTytV8dmSAUEN0MuuD+1sWU0zYe5CFwTjYBhxRQWQznd//273HQiTd47fOkbsFLse9eueICHWnxBP00BC0eOoTPK3/CYv7nIlhjipmXDO8a8Wrl18IyOFYoysUHMaVMlXyAvMDYgozGZtfWSkI86uTNKUxUdEkQOjS01iLPytdXyBeoUSHOTU0L4sVRQObqv5KMTm1pQS9JBaPybirUfp0/DhkabhxKTfTb9lRf/GH1xYczbaUaMnz/TAaFDbNcJXnK4OuRAXz0zAGs55MxcpAj9MB9fnwhxuvS3ytRdcB+SArPfFvna9Y/h3Mltvmt4Ad8pORv7dCeCklp9ODaJhhu5x9Gnn1HWnPhj6I0S/gpnqO/a9x9E4oOb9SilNv44J7OQRHAzp1Fp9ce6dmRoOohfVFYaZ9xje2CUyQA7tnmCIcxgAejjVeBfXN7q9WiScp7l13g2OicyTw3ar4sy63z6/B8e+Uq+Z+ieAqOrTGeKgKkXb0dHUq11/+s6/tTnzHRLQmWXPKEl3YqjWQSBg2pyqtXR+e/8f7z/8/t7xzPGHAWk5Jzid7da9Hvqhuf9F63qihzrKsao41x7tZJw7M/ZOYLXx0bVE3fMe7NxuLvcVpj7p+IpbYskTLah3RudsxE9UJHKFOyqycvM8+IWOMenNdSyXj3D/MopcPIvgp/7Gx20sdOUMLbcHgrte0Lb1DAW4XQW5BC80Dz6BQcbaX/AL4ekdnaxxAaLnw+NH+e43l369YfiuHsdrcQEFcRdXKaNy780U4fQY7j2aE57YR5Xl5Qn+O+rIo+lBxP+JDnqoikshmQ7i3CyMwVFlgumCc1GUuA5UvbTG/XftWGJCgt2jNlt8TLpwMvU34dOpzwYMXzqamOE4mDz/ExQlEPiP/VpNILSX1Vw0uz2dL9nIOA27Bdn4k14rOokCc5RmqyfT88lxS7JFGuDgem+O1tsY93hgeQozeI2kCf6Q4NUzial9wErkuCGYJJD8vDHbLrO49kdPMVv7F7WWlBYBrxIGFAibN06BBxI8oVfQu7NhjlwaC7gJzQVxl3NQ6EYKEbyh0F4Bd+MFNdBL/iHd0cQy092KK65p/UUA01GsAj+hbkIYytC3CwtQlQSY4d1idfR4EAqFiXW4CTAcItq6lB+BRfM8VUIXp2kEVhEG87kduGI1tYJXnncS9+4cFekxS2Wq4zIw8hywtBMp8XptIbTBPdhSDqCjoXgXPI8AlAHoeohTwok3n/noRi4M96GH4Drpo/XtQzPeo80cHhb/3fHjpjf+tlhnTR3aYuiqI1JgBeY8nhrd12tPNnWhBwQvKtsBAh74B5hGy3s+2S6+z3b7XciD0WxXe4i524X3RxP5/WCfoE4QIlYopvtEamIwJJ9fhk4MrR0IH6jSjQ61qTIIEyLLba19r/vSZSgjNRQT1s6Z6ZKgKlS9jaVi5AdWSjUdHl39vehA07jmQsrRZOr9N7lEgdcr88S12abn2h5FUi2zdA3BsDdx2NO+LmA0rE5iqlgNXebPV+SYAr7klB4FhqAYzrnDE14ViHd2RZZ2GpQo3HLVE0JGh3Toc0RYzNsmfhXYL+b5ypmrLCFHdJ5vH7ecuT214XfZg57XdXwuEzqjSFtdjo9mYis8BfkVwN3JzbaDejaA9iJ13dB+fRJ8e9qz7cg0edBHF/GAo7zFnPUSy0J8fuhR98kT/vjkyZeb2S2ur2qmynJvKS+PTXRanjbI6ozsI8C+9Xm1r8np9x7a4alTtKulC96WrwJ1hYSvy8N479wMhKFxICH0db3LugaDHo7PdPWvOu2ZFxj8p/1dQBqPUsA8lK0bl+EpevRT5qAWKZhZcFXk8QYhnkEI0YNcevO28eJSHA5oTg+gSru3R51hTudq+9B1EdLocPCePJmCAO53rudd5ueUUoW3/VI6ptWsS7C+GvvGUWlVhyD2ef9QifZ8a3mBCe0cPSNeJDqZjRbZpUIovUiU9ngGDiidj0AvVJZFsqnDZ8/9EgbDXHn40t9WqN+e+rzSVPcLb5msoZOUe6fNYFpncskkaJ+0lfo5orCACEyzQTjklnMSXyduPPCRGUm/BegZTMxJG/DDd6T3mX/oOXidxrvfkTUDkDMYGewSIIFz2lZK8wK7JPEB5/iMIRG5Qp0WN3iNnJeuAj4M1Zl3ngus0bd1Ms6990qWCBzlzKGNz9AvjLVXVRGaV3RWfIm1aMxtoBlXjneJbyKacU15sksQ460Mx5dqe9PMZtGid7tjzAOGQCDxCBOQ1cc3Qs+pr+HRVhh44K4aIL5Tx4vJCBO83g5dstnnN5St1/sFdbnNGpeg+MjSE9pUhc+JzLYEh755FwZrYoG0LcwtY1BcwEoFCzAx5gbSDLgdVPzJdBNaM1mVCPdc3wX4HusNSwXRPOeIdXbvLQZ1kXm6vKeTTpYmGLrpclvUdchKxtBIoyMdZaifI3g8YYTEB/EbrP3io6PkdfCrc2xw1Bc0Gc6vw5013NEW5TGlXIJ6kXsBpi0pZhpCjYxBHuybbIvOc1bSEkbojUXHkv08GB8D6VigbmRpjHKLt16A/emZMZS+/lrpwoA2G0fM7szoD8S9joLlzNSPByvoUS/FDySt4DyPXvZMFwqxamXJr77NhOaK95PIKM2SBq2IkmrbQxT09jycWDEXvnsVarU+W1YyvunVHXZjWnsorIbo5HHG1XZlVaATL7hInOocqeLT8VrbjmW7kYGsAby3wwZhElNWGStkZOjB5gq+Dpos0/q0QYS64kPt0Fup/M7d+EY3gwc86WcgRem9pdj1HSNtWWIPs2GBVmkUkGZDcTB0zYReABN1vUDlCI+KfhRPmsdjgEHb7KjCVD0vfH3CckZDD+cpB3K/aMadGa2HgwuBKUyZPthlfIADzBLjUMxdZmsQU51912XuRi4S12TPto0Ot4HbC24C1f4LUOb6Soyojlcy0tfxuPf+QS3XPVqjuCZzAa/N88CXB/Pbwfp2ZtRi3Hs/PKOCiRKEK8VHiuaMVwf1SucW4XUeW6X/bHvb64IIQRJV4Ows8brqqpssiZYHdeofloBLz33modC620bNsAyxXbCTFdhmam3cumBvNmlW7M78+N2FFScX+cQNFb3jcAu/yMN2XOWSgADCdPGwREQxiUaYq2H1kbYw5L/ev5bWRoF1uA0gW1tsLnEbybvPafk17bW1+2xzxY+KFW1SLAY33pj+YY+Z1gFLg78e3K7phG0drhXGl4QEPGpk1qc3BIyko9yDJt1G96EecsTbgPyAPlZew9x3rkNR7XHPQziMo4IzuNt0Gqq8L8PHEQhuMRmfFqtw7F1opM2XKEND71+N2W0wJEUvcmc/lT7hhTYUir2UM9psxWj5JsoSZ8aLeIKdHU2ptnlvsds18FuaeqdgtTkGIFRR4/4apSjA4t+XEIU7M9Qe696GjaWCcb5a+JdF0QC0uIxIr7kwhyGpxzKzOpyMx2N/iZfjhM/Y5k2mT43qya6DKEe8gUwFjoDZFk9ioJ8ifsTDoDBrfBJ0n1xX/7Cqij9kLvDqK6GME1c8/lxsd8behYipQEY0d4WtnDEvEldZTRfgWXfsYKV4mCVcMh5RIYAqHJ8Pi2pbQq4Eui0gV2uoizJyoB5QLpy7R9dm2fkhPMRz1GI4G6Mqx+AjxDG05/8NlZ5qw1sduGWsBsoyY1JYSAV92w7ZuOXRa59n3p4DpQHKKtvFeFYeucWQWqxLAX5wCcI8Xoz4x3LhdZdXmAfMup5/Ea/X60piik7QCSI8JaxdZwrea336+k7Km5mIbwtw7xrwFBu+UglbxLW1EUYxV2D607h2jOSJo8YQ+/M2pccFEArP/tHGTvtRLoAhWy51/gPYZLt0mtaxUzF6eZzZyvkSP4iR+LJ8cHjvtwZXxidFR6WB/T1UNR6puNzUZo+6S86Pb6NtcQeguS0+b7L1pntRRoBdVdxHIFJRneEpb/Xtwbs88g9RG4DFJTeH3SqZU62l62JYjRdTrjHuGfs516m62MMKk/MLUiSeZ+2lnfzvnKJvygRwvzkFAfTPbOF9lxN7YSqbhxrRJsYsBw8hvs/q0Ii+0ioGo86EnOuZqLWnthOLOYkswPTAngmez0QDme+32051CnXZB+i5OGlINMQ+T4l7qwwP/egLLaXX8aQGFhI+Sl0qDuzECrM3KmGHbh/EsFSgr/p6r0J8tPp63vnEuyAqnJukhfX3cc1xzf6xj2FFQQPS2P58HIynz/1x8P1fny/M1Po/wbYWTieZs2VMxK7jSvx3mCeRIMq0FVz3Z9g1wle7e2g7jiEMijOIOt68Qo55taYTDsz7Ebd2Swj3qFY2y1eKpHx8F/MdZUBbUHxBmc8dvctqn2NwqqJgOvbhc4LGOrmC49q15shL9g1D2f1iIOmiB8AMlPqNhbHuJPAVLqPP3hPu7V1qdPExVM01FecYIe4iTSyIGXXrrkV7xg/DFWjhXP3nr+8+fHx9/eHj34VruAqeQ2dvlIttORGsrpyfP7x78+G3a+FiTqfabw8gLE2xR8XvOccHQIywrN4VNxKkaVeQycIrCTDiz5Z0CylH/Co0qw95s5ENFlhDczeP+J6IcPKSHZfvv+/8FpzK2x/fXr/7u/j09/fXP19dv/3xUmiZ3pUZH8pT4iCu0fSJ1T5PuMIiw/su8C4bdectFnHCvJpN8GfOaXDx7HzlfHr7y5dsNn6WPjjd+SJ1QkDNQrkHWVqHX5KZ67x9/2ZCZM9+0E0uLqaK3PBx6qhTXHzepmNALp3mIx1/EVd0Fw9f8NWevVAHc6k/CjPwDbC7Oud1YRctE/0DVcPb2MWJB3B1/R0w8abmw87T8eR7f/7Mf+6/8P/qv/S/X4CDhJetqpfq8Zn9+IIeF+ahZ8IMYTJ02/SyI6BOH5V4jL6REVMTlBgi9TD6spuNp+kD3buqD/g99VfgMIXO76OfPr6l+ikuUPaM2Rm3Yagc7+VSrkP6Dd7ESBVx0U6jS6clwYFuc/l0LhyaLA2UoDtllBU4lUXPkw1lq3mjF4wqqOvxX32VsEY4XqvHdIH1F2aKWb/fxO7mM3t8rdmDOrxprGfCJ56Ybzuax3SW7+kTwvp8HDx/YtS/A8PSEYrF4vxoPt8PzEePdbeRuV9V/pKc1D+yUrfzY3yLQ/WS62lntdrCNvzXN0+Jzgju0FUXyumqqpOW69RdGKrncgmktS4locHOLZbA2zyAJ56j2t4jQ03Gp8c7fWXFDPMJD9rsQbCAjtnYqAo+0KUWyFK20JxYQvpsMxyzQ7vv5V1yTK88E1y+l09W88lihD9fwM/p8Qo/7Z8nLO5CysHiera10Cs85QOf9OmdLl0IszoKwQ/H5LLuB8YmgzR/4fl9xtApqVnLpQPAjRs1j2GTiD/1zEuWTjQaXOjhjNeM3x7f80TLPgBl8K7lk4T4CvOduKEZwQ2ot+8HQZy6y/kEkL8OwRi4zo7n//gtcadodPoWNgX15I1nprB193UpZQOs6qH06U2OoeQLuIjWc3toB5wyygo1B7xTtlH5GE7EkorpHJZf1J4lR+EzPmFGETpfJAheKgoeNUno2nt1CI3PkpK1bp2TMjy62dm3rrDifYvLNDx5A9ZQez58GlfbDDeQ7FvT6DDO5Sqr6AvfM2Z8D0Pdz8rF8iVGftQ7dEBg/LQtcfHt0xRP2x3/MNV5UnigUfFvZw9e6VEHTcICzNrLly81sPZyHtz1H7icZ4SvH7mchy5GOTEX6PqV2eD/FUHVKIqEIQPT10jqWyrUoTBjzR3rPGKpyaFe8AlH9ZskhArSB/XJIny8k3VJ+6Di0X7QYWbdLK5AneYoUI5lYxSazcpY7f4adWiviT4wvjrVoyijr9LsLurAznhNDWLQ3k3zp8gzrCD/CfqwgfhfJQ/dZcnVc90B9/Zykn+eRobV/BphrBxsS4P/eRK0txrrm9q7a6MH5k9nzIZOQH2bMqry1MeTEqc69/5PFgZhIMnW+hIGmc6so7N8yN8+zSvT/sFZGs9rkylfgVd79nncx+EpDdU7Y0tI0wk9vIiAfuepN3wclxBSbfUDNvbtk7Lq2J8AM75q9JWfbfUmGciIV9CZPQdthjgq3wwvC9A6nwJDvK7IMY9H8mYK49ArhDx99gViAPj1cPb/ASHdviI="))
assert hashlib.sha256(_data).hexdigest() == "033fabc6aa4477c24fb8bd65aa9ad9e68f942ec147fbf598c8438482aa522650"
if _dst.exists() and _dst.read_bytes() != _data:
    raise RuntimeError("V3 extension differs from this Notebook: " + str(_dst) + "; compare/back up manually; will not overwrite")
_dst.parent.mkdir(parents=True, exist_ok=True)
if not _dst.exists(): _dst.write_bytes(_data)
print("Loaded V3 extension (V2 source unchanged):", _dst)


In [ ]:
import base64, zlib, hashlib
from pathlib import Path
_dst = PROJECT_ROOT / "V3_extension_modules" / "v3_local_adapter.py"
_data = zlib.decompress(base64.b64decode("eNq1Wm1z2zYS/u5fgTIzF7KVmSaZ9m6U8kMuVlLfpXbHcXzT8Wg4EAlJqPlWApSty+V+++0uABKUZLtp5zyZWCKAxb48+0oHQXD1ki1ama8EW9YtO63yTul2m747O0svhBK8zdbp+zrjRSybbbVgYd3Klax4wa5esA1vJV8UgmV1pVue6Sg+OjqrNePs59NLlolWy6XMuJZ1xXRdFzG7XAvW8ls8nXPNldCs4NmNYk23KPqtshRK87JR8dHJ6et3Z+cfLk/fsLarFOOtYOKugb1SF1tWwW0bXsgceVjKtuS6brdMlI1sgVzBxEbmospEfBQEwdGyrUuWpstOd61IUybLpm6B3wro0N3KbGm4Xhdy4dZ/hq9H9vOvqq7c5zVXuM19rbqy2TKuWNW4Rw2vcngA/5r86Ci9epm+nb2+/Hgx+8ASdh20QqfPgwmjD9+5Dy++dZ++p0+bugDmCqm3dmkhNB/WulKk2ZpXK4EkjtjOT8DLuqu0t4UFIH5Vb0SbtlwLfFDUq1SDDoq03OD3ZoH/S4eHViAHG5Fa9uZH6fvZybvZRfr2dPb+xAizFJzUyjdcFoiLlOsRFVRzB7bI7UIpyoVo1Vo2e0u5yKQCe9DXPZFAKFV3bSbSRuqUd7nUIsdja6nA/mj4VPO7uqrLbQpiAgrNOrAi7lLvXn9xCQiuQSUAGbq6X5sfwU8uloyebOlSh6uwEOA9bYqImQCmZbG1n5sWNFzxCrmEB9GUxAAUXiFeQe+ArrLT5ECOGiFKAF6qnCEf4la0LFuL7KYAydhCgJeCuxVclrJaMSs5K+scAE70fyQCz/ozxmMUX4pVx9tcTdjZ+SVDPTQC/qs08lkvmV5zDf8J1sHTttgCeaJ32C+JqG478QrYNGECGC3BzPBdc1kx1TXoAMgl+nsrshpuj50ODG1Qi0o+BahPq0a15i+++z6Yosf5qo0mtMvod7RpUHn0maiW+DmhtV0T0LpcUtSgbbFU6VIWIrTWwZ+WSyUYGKkTs7at2zB4/fHk9HJ2kp7MrkCQ3zoJMlOEMyD0LM3+8eH8jN1KvXbmy3vbBub6EvSTYBCJi5rnKjR8tILnqRZ3OoStdQ5qS4JOL4//FkTmGIZnAGORT3A/GNBoLwb7l8pn34q3I92eUMvgJ6kUmgcF6fGH26fsE57+bBnGn6wumw6QltiIFxsThAPvi60WwEcUr8VdLleAkzDyeUKx45XQIQkRfZU4kodYQ5Y8peKlrJQKgnu2Bu6IhGNvWfCVSq6HKGMjA3qYSp0VgokfbTxQ7+3zg00f0ixNG9lMKPZIA7YB7EAohRCCWSRtagkhV1Yp+sx91DVvQSMpUBQF8ARfhp3zweogINqbBJ0e1iksRUwqsvwluOV9Sh3HFGISFYs4QL0CGadWCyOIhBW4PAa6/jYXex1oUtgoVBBNQP0RA44LUdHu63t2zmPYKRtAyw8vvn3I8XZwYOPLUrSEVcj4GUQbwTD8QHLKnT/ueBzsgXRH2rJxnOe80RASKlGEP78+m72fQnqOT6AgedvyUkzYm/Ozt6fvpiyXmZ6wryeQwFXXIhNJMFQkkDS8EJWc1ZUYovyFiYN9vdTWt6nMJ+z5CwqIJtlAOFZQy8AdGebxykT+UgAU2I3YKhvUDwYgL4ZDIeSSFXB6fFPVt6NojTCaXc0ufmGQHrObZ4h5E4qxHjMyGO0uBFuJSiCWc7bYUkbwIjwVR71IVt026Qob3f8lWF6bEMurDuUED4KS0F4DrPBO18clb2+Y1D3j49RAF8Wu/tq8BMdQMu/gUiilNi8dRHuzGLBW7NPYPH7oDj4/BDUohAdiThVe8Qls+7Rs7DHhQUH5g7k2NLDpKyFAOpR4/DaY9z7Vn/gqYX41OD3E0gumb0WxEccGLEyBDUrOTCGXv7LezIB1wPa6zsHQslXaMtcAVwTuOKubbeh8gdCTw9qnwEAS4peSK7AnhUL4plUKKQg/ETbhd/HcPQGwQYi1G8UdpWT8vF912uAm7jKhML6iDyIpCnZ9iRhMvnYq+ex0ZAMSmyYDu8cMWoWweShNL4NzB0wSu6fTEwGFWjeDYKcAWCIP7abIi3tNnHcN5QdYv3Zamkcxr0CNiISdHYfV5w48hLqTngwyZ9jG2IV9E+SqgaewubbWmLMkYaD6rOhyyBMeV+gCpKWYdsZdJX/rIPlH7Ae0NnRoEi2wMeUnpClkFkLEw47xsYJ2S2SWRaLMKgiSjjmMLRm63o4e2AgrzAPLfLiuuc5AHgy+uqZFDFohPo32XRzl9n1wr+bxgrHTx14deaji+4KqjzNKmseyOkZWR+WxDXDhmw9X0StUlheBM2oxMabICvJXi5br77cHSQ9UTGVqM2J6lPQH7wjR2uNGLGL/AVPvGMIh8nNk/cjQflQDmH5dcnBFQsC+AW00bbjjP9GuOGM4HLDywW0jUc2OP+Zrj0qEMwtwCMV68iY5HlNrhtiVPcYdFOGKOyq9Y5euEf30aMcMjzJSCK4wtZZQ3TBeLuSqqzsF1Y4AR1HaFebuHtv5YVdFvZnLvUYYj02M+k1MFYTT8H1Km3w9Zno+P9RpM1ZXyb16n6zr2yQoxBLShA0sUCJBIZSCqeHXWIHN9c6N4I0VD63xvtiEZnI09C6YImtwyj6U1gsl2o0Z7Hic+CHrwBjhz0wRHhgijAVzbRqEurjkTVgABnLO7qYIJ0VFPMYwKJTDu8jVy9AyQvEeRkkSYPONwb8odlV2XxjvK0RPg9RTkD7Qr7NoRMgGZ2SFnotCieGqJ1g7KkGTgPOz97+wrKjRHEgMswqCFKvHBgC8gC+vWCWABWhMlcIFqCOXGN0dkv2hn7uDXBHwvBMk4lzHVd2WgLd/C6/NhH3+3AiZJwrfYOi5BPq5KDQP13ChSp7/dXTw4PDqEQrfjygcHnP1JI7HJHK+VcnLb0cU7puHPUjj+YjEATzj6bccbOfvexDiB0/cD/qD2+/3g9FuqOypFYWHm5cxsWz7smbiCsPEfZjYNAxUeZEMLVkS7JfmTzAK8ALT6ZZRb60Y1Z6Qi9vapNClvMOmkUqaBag75xjzX0HnJRpoTZRdUlB0Q0WWWWA6lq+DBmSqjUTDQ1ulkVO7omt6qPiaHqjEpgDgjSjqpoQEBPna+F2VmykFFu1HQ4FtSDPXdNDF1HLQSgrHAi+ij67ePzOs7h00nB24Bp77m00BbxW5vV8pN1OLYnL3cBNRzLiZbDAk98K6yZZVwkDLdBCdovZhrPvd5mIec6W3jQgXdV1E7C/eVq8cjQvcYLgeDT/cbgi72XX/xVidQjDZlO1wZNPYgRh8VjM6Y4EYfMllPla+6Mbh4PhaOxVxN40mI9ibCzttDX96ffHP2aWbhVzM3s4uZmdvZr9jOGJHs+TSe/ORDw1vgdtVyxscZzadNk3/+cXpu9Oz1++Zuff6KVF5OqfJSH/79dMF5Pw1Mvp0bqckHxWOxmslnkEVldInUInGuYWGlhlHEACHY68oAHF/TW1zHd6uZbZmJd9ilU5xS+TRMCXJ6ta8BYFjlmmF6RmUUGyB8mplJiYc+xC4B8omHJHznKo16IHxIAJcEkdmZP+nRh6P9kNj/Y9aovFSFFPRqf5gUzSeRPWDb7qDmbEjYQ3f+HllmFkfuh4ABPSrepc3lAbLoD2Wq1zhpD18GtuTTyOqUUZt1M6podF18LdJx+wzT5U5NU43WAb1DhvQhsBNnQxUUcFBD8t+cfCYB7rsmddj/37gW2WiXhN7ynI2h6p/mOMgDIWr3ckx4HfvJMF87o+IiH4yXOiJNNqHt177pA/0ePt7vDseO3xgU28AJDxuDEcy7vaFaBoi55/xj3zJrIYNxZPF8G9drc3ANvB1YzBlXqvaAdggZtVBkyYzqyJriInAu1QSZLVocYDNnh3c7tlu78jx8x6lwzYEIilgkNCqt7fuyAUOMGr333fts4O7fwej47bCoM/YCkcM6QZNoMbG+n/JEDeZe08eLmVRpGauajJXb1hTjXPbZBuSO/i7j6H5BJrp0daD/TPUm9txA+3fm7d1A12z6iCR6eQgzO7nIPJd+AmDAkHi325AK+F3y5jbMNFApzGFRAYQ72haQdERGjeX0LwM+PfZ2/OLGTONGhveMBjAHmqrdiLDTlf3zQM9mz3sSUwR/p4e47FQr/k40tPJyBZEZkKTKl4in/2xFErWlMoAkyum4/dHXruqbFB5wt7gBMeUDgZXOBWkV/NWv73utkbnEq83gyIaNoJRxm9LnrAfRTtMkQp6ywT2qXBU5ZGDEg3nXda8sBWAdexqEihdektRj0EzZBKH4vApNn7hSCLfTqNqknRBFex/zcfBwDvmjfHNYmjI9WnF/pXFyxTCRk4OYCf6YbOjYCplpwC+CaN2cagrfxKgVCibj7HCKkAYkSum690Xcc9o+dmWrrJvVuL+DxPgWRI2JEgT084koSvn8aqtu2axDfu3J1yl1Btb3FwHhiZ2O8CKUxA+gg4fSQVzQ6t/3wBLsaEWV25o/1VCcu09f/AFyCei+3lKJCW9ijaIcC/PBy3QgGzcChCL18NbIcPspJdnfvQ/NtKg3g=="))
assert hashlib.sha256(_data).hexdigest() == "d1d355da4f741afde1228b1507d3c824eb66b91c4b2515018715692368ff403a"
if _dst.exists() and _dst.read_bytes() != _data:
    raise RuntimeError("V3 extension differs from this Notebook: " + str(_dst) + "; compare/back up manually; will not overwrite")
_dst.parent.mkdir(parents=True, exist_ok=True)
if not _dst.exists(): _dst.write_bytes(_data)
print("Loaded V3 extension (V2 source unchanged):", _dst)


In [ ]:
import sys
import industry_gnn
_ext_path = str((PROJECT_ROOT / 'V3_extension_modules').resolve())
# Prefer exact embedded V3 code, even if an older standalone V3 module is present.
if _ext_path in industry_gnn.__path__: industry_gnn.__path__.remove(_ext_path)
industry_gnn.__path__.insert(0, _ext_path)
for _module_name in ("industry_gnn.v3_residual", "industry_gnn.v3_local_adapter"):
    _old_module = sys.modules.get(_module_name)
    if _old_module is not None and Path(_old_module.__file__).resolve().parent != Path(_ext_path):
        raise RuntimeError(f"旧版V3模块已在此kernel载入: {_module_name}; 请Restart Kernel再Run All")
from industry_gnn import v3_residual as V3_CORE
from industry_gnn import v3_local_adapter as V3_BRIDGE

if V3_ASSURANCE == "AUDITED_DEV":
    V3_PIT_RECEIPT = V3_BRIDGE.verify_pit_evidence(
        V3_PIT_LEDGER_CSV, V3_PIT_DAILY_PARQUET, V3_PIT_PROVENANCE_JSON)
else:
    V3_PIT_RECEIPT = {"level": "UNVERIFIED", "confirmed": False}

V3_PANEL = V3_BRIDGE.adapt_panel(
    PANEL, CONFIG, assurance=V3_ASSURANCE,
    ledger_path=V3_PIT_LEDGER_CSV if V3_ASSURANCE=="AUDITED_DEV" else None)
V3_DAILY = V3_BRIDGE.adapt_market_daily(
    MARKET, REFERENCE, assurance=V3_ASSURANCE,
    daily_pit_path=V3_PIT_DAILY_PARQUET if V3_ASSURANCE=="AUDITED_DEV" else None)
V3_DAILY = V3_BRIDGE.remove_same_day_returns_for_graph(V3_DAILY, V3_PANEL.signal_date.unique())
assert V3_PANEL.row_id.nunique()==PANEL.row_id.nunique(), "V2 row_id coverage must not change"
assert V3_PANEL[["signal_date","ts_code"]].equals(PANEL.sort_values(["signal_date","ts_code"])[["signal_date","ts_code"]].reset_index(drop=True)), "Row identity changed"
print("V3 panel (compatible):", V3_PANEL.shape, "daily data:", V3_DAILY.shape)
display(V3_PANEL.groupby("period").agg(rows=("row_id","size"), dates=("signal_date","nunique"), eligible_labels=("label_usable","sum")))
print("Assurance:",V3_ASSURANCE, "— original V2 test period is reused as development, not new holdout")


## 11. Complete、Sparse 与随机稀疏图构建及前视防护


In [ ]:
V3_GRAPHS = {}
V3_GRAPH_AUDIT = {}
for V3_GRAPH_MODE in ("none", "complete", "sparse", "random_sparse"):
    graph_samples, diag = V3_CORE.make_graph_samples(
        V3_PANEL, V3_DAILY, graph_mode=V3_GRAPH_MODE,
        feature_columns=CONFIG["features"]["raw"],
        lookback=V3_CORR_LOOKBACK, k=V3_K, graph_seed=V3_GRAPH_SEED)
    V3_GRAPHS[V3_GRAPH_MODE] = graph_samples
    V3_GRAPH_AUDIT[V3_GRAPH_MODE] = diag
    print(V3_GRAPH_MODE, len(graph_samples), diag)
print("Important: little/no feasible rewiring makes the random-sparse placebo uninformative.")


In [ ]:
# Automated guardrails: matched degree sequences and zero label-to-graph dependence.
from collections import Counter
import torch
for gr,rand in zip(V3_GRAPHS["sparse"],V3_GRAPHS["random_sparse"]):
    n = len(gr["ts_code"])
    assert torch.equal(torch.bincount(gr["edge_index"][0],minlength=n),
                       torch.bincount(rand["edge_index"][0],minlength=n)), "random graph OUT degree mismatch"
    assert torch.equal(torch.bincount(gr["edge_index"][1],minlength=n),
                       torch.bincount(rand["edge_index"][1],minlength=n)), "random graph IN degree mismatch"
    for g in (gr,rand):
        src,dst=g["edge_index"]
        ids=g["industry"]
        assert all(ids[a]==ids[b] for a,b in zip(src.tolist(),dst.tolist())), "cross-industry edge"
assert not V3_DAILY["trade_date"].dt.normalize().isin(pd.DatetimeIndex(V3_PANEL.signal_date.unique()).normalize()).any(), "lookahead: same signal-day returns are present"
q = V3_PANEL.iloc[:0].copy()
firstday = V3_PANEL.signal_date.min()
q = V3_PANEL.loc[V3_PANEL.signal_date==firstday].copy()
q_fake=q.copy();q_fake["target_excess_return"]=999999.0
base_edges,_=V3_CORE.graph_for_day(q,V3_DAILY,"sparse",lookback=V3_CORR_LOOKBACK,k=V3_K,graph_seed=V3_GRAPH_SEED)
changed_edges,_=V3_CORE.graph_for_day(q_fake,V3_DAILY,"sparse",lookback=V3_CORR_LOOKBACK,k=V3_K,graph_seed=V3_GRAPH_SEED)
assert base_edges == changed_edges, "future labels changed graph topology"
if V3_ASSURANCE=="AUDITED_DEV":
    assert V3_GRAPH_AUDIT["random_sparse"]["random_edge_change_fraction_mean"]>=0.10, "Placebo graph barely rewires; no identified graph test"
print("PASS: point-in-time graph construction, stable labels, degree-matched placebo")


## 12. 训练 Ridge、Shared/Industry MLP 与残差图模型

所有神经模型共享训练种子、隐藏宽度、MSE、早停和日期切分。


In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

# Immutable cache identity includes ALL V3 input columns, causal daily graph evidence and settings.
# Reusing a checkpoint after changing data/code/graphs would invalidate the comparison.
_v3_panel_key_cols = ["row_id","signal_date","ts_code","l1_code","split","entry_date","exit_date",
                      "label_available","target_excess_return",*CONFIG["features"]["raw"]]
_v3_daily_key_cols = ["trade_date","ts_code","daily_total_return","benchmark_daily_return","available_at"]
_v3_identity_payload = {
    "prepared_key": PREPARED_KEY, "mode": RUN_MODE, "assurance": V3_ASSURANCE,
    "panel_digest": hashlib.sha256(pd.util.hash_pandas_object(V3_PANEL[_v3_panel_key_cols],index=False).values.tobytes()).hexdigest(),
    "daily_digest": hashlib.sha256(pd.util.hash_pandas_object(V3_DAILY[_v3_daily_key_cols],index=False).values.tobytes()).hexdigest(),
    "code_digest": hashlib.sha256(Path(V3_CORE.__file__).read_bytes()+Path(V3_BRIDGE.__file__).read_bytes()).hexdigest(),
    "corr_lookback": V3_CORR_LOOKBACK,"k":V3_K,"graph_seed":V3_GRAPH_SEED,
    "hidden":V3_HIDDEN_DIM,"epochs":V3_EPOCHS,"patience":V3_PATIENCE,
    "learning_rate":float(CONFIG["models"]["learning_rate"]),
    "batch_graphs":int(CONFIG["models"]["batch_graphs"]),
    "training_seeds":list(V3_TRAINING_SEEDS)}
V3_RUN_ID=hashlib.sha256(json.dumps(_v3_identity_payload,sort_keys=True,ensure_ascii=False).encode()).hexdigest()[:16]
V3_CKPT_DIR=V3_OUTPUT_ROOT / "checkpoints" / V3_RUN_ID
V3_CKPT_DIR.mkdir(parents=True, exist_ok=True)
print("V3 reproducible cache ID:",V3_RUN_ID)
V3_PRED_LIST = []
V3_SEED_PREDICTIONS = []
V3_HISTORY = []
V3_PARAMETER_COUNTS = {}
V3_FEATURE_COLUMNS = list(CONFIG["features"]["raw"])
V3_TRAIN = V3_PANEL.loc[(V3_PANEL.period=="train") & V3_PANEL.label_usable]
V3_VAL = V3_PANEL.loc[(V3_PANEL.period=="validation") & V3_PANEL.label_usable]
assert len(V3_TRAIN)>0 and len(V3_VAL)>0
ridge_candidates=[]
for alpha in CONFIG["models"]["ridge_alphas"]:
    model=Ridge(alpha=float(alpha)).fit(V3_TRAIN[V3_FEATURE_COLUMNS], V3_TRAIN.target_excess_return)
    v=mean_squared_error(V3_VAL.target_excess_return,model.predict(V3_VAL[V3_FEATURE_COLUMNS]))
    ridge_candidates.append((v,float(alpha),model))
_,selected_alpha, V3_RIDGE_MODEL = min(ridge_candidates,key=lambda x:x[0])
PRED_RIDGE = pd.DataFrame({"row_id":V3_PANEL.row_id,"model":"ridge",
                         "y_pred":V3_RIDGE_MODEL.predict(V3_PANEL[V3_FEATURE_COLUMNS])})
V3_PRED_LIST.append(PRED_RIDGE)
print("Ridge validation-selected alpha:",selected_alpha)

V3_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
for model_name,(model_mode,graph_mode) in V3_CORE.model_specs().items():
    sample_list=V3_GRAPHS[graph_mode]
    for seed in V3_TRAINING_SEEDS:
        V3_CKPT_FILE=V3_CKPT_DIR / f"{model_name}_s{seed}.pt"
        if V3_CKPT_FILE.is_file():
            _checkpoint=torch.load(V3_CKPT_FILE,map_location="cpu",weights_only=True)
            assert _checkpoint["run_id"] == V3_RUN_ID and _checkpoint["model"] == model_name
            V3_CORE.seed_all(int(seed))
            net=V3_CORE.ResidualFactorGraph(d=len(V3_FEATURE_COLUMNS),h=V3_HIDDEN_DIM,
                mode=V3_CORE.model_specs()[model_name][0]).to(V3_DEVICE)
            net.load_state_dict(_checkpoint["state_dict"])
            hist=_checkpoint["history"]
            print("RESUME:",model_name,seed)
        else:
            net,hist=V3_CORE.train_one(model_name,sample_list,seed=int(seed),hidden=V3_HIDDEN_DIM,
                epochs=V3_EPOCHS,patience=V3_PATIENCE,lr=float(CONFIG["models"]["learning_rate"]),
                batch_dates=int(CONFIG["models"]["batch_graphs"]),device=V3_DEVICE)
            _tmp_file=V3_CKPT_FILE.with_suffix(".pt.part")
            torch.save({"run_id":V3_RUN_ID,"model":model_name,"seed":int(seed),
                        "state_dict":{k:v.detach().cpu() for k,v in net.state_dict().items()},
                        "history":hist},_tmp_file)
            _tmp_file.replace(V3_CKPT_FILE)
        cnt=sum(p.numel() for p in net.parameters())
        V3_PARAMETER_COUNTS[model_name]=cnt
        V3_HISTORY.extend([{"model":model_name,"seed":seed,**record} for record in hist])
        pred=V3_CORE.predict_one(net,model_name,sample_list,seed=seed,device=V3_DEVICE)
        V3_SEED_PREDICTIONS.append(pred)
        print(model_name,seed,"parameters",cnt,"validation_MSE",round(min(x["val_mse"] for x in hist),8))

V3_ALL_SEED_PREDICTIONS=pd.concat(V3_SEED_PREDICTIONS,ignore_index=True)
assert len({V3_PARAMETER_COUNTS[x] for x in ["res_gat_complete","res_gat_sparse","res_gat_random_sparse","res_gat_self_only"]})==1
PRED_SHARED = V3_BRIDGE.v3_pred_to_original(V3_ALL_SEED_PREDICTIONS,"shared_mlp",V3_PANEL)
PRED_INDUSTRY = V3_BRIDGE.v3_pred_to_original(V3_ALL_SEED_PREDICTIONS,"industry_mlp",V3_PANEL)
PRED_GCN = V3_BRIDGE.v3_pred_to_original(V3_ALL_SEED_PREDICTIONS,"res_gcn_complete",V3_PANEL)
PRED_GAT = V3_BRIDGE.v3_pred_to_original(V3_ALL_SEED_PREDICTIONS,"res_gat_complete",V3_PANEL)
PRED_SPARSE = V3_BRIDGE.v3_pred_to_original(V3_ALL_SEED_PREDICTIONS,"res_gat_sparse",V3_PANEL)
PRED_RANDOM_SPARSE = V3_BRIDGE.v3_pred_to_original(V3_ALL_SEED_PREDICTIONS,"res_gat_random_sparse",V3_PANEL)
PRED_SELF = V3_BRIDGE.v3_pred_to_original(V3_ALL_SEED_PREDICTIONS,"res_gat_self_only",V3_PANEL)
V3_PRED_LIST.extend([PRED_SHARED,PRED_INDUSTRY,PRED_GCN,PRED_GAT,PRED_SPARSE,PRED_RANDOM_SPARSE,PRED_SELF])
print("V3 models:",[x.model.iloc[0] for x in V3_PRED_LIST]); print("same GAT parameter count:",V3_PARAMETER_COUNTS["res_gat_sparse"])


## 13. 训练 HIST-Factor 组别

论文核心结构被保留：市值加权的预定义行业概念初始化、全股票相似度校正、隐藏概念发现、个股信息残差，以及三路 forecast 相加。差异是本项目使用周频 12 因子 MLP encoder，且当前只有行业概念，没有主营业务概念；结果必须标为 `HIST-Factor`。


In [ ]:
from industry_gnn.hist_factor import prepare_hist_samples, train_hist_factor, predict_hist_factor

HIST_SAMPLES = prepare_hist_samples(V3_GRAPHS["none"], V3_PANEL)
hist_code_hash = hashlib.sha256((PROJECT_ROOT / "src" / "industry_gnn" / "hist_factor.py").read_bytes()).hexdigest()
HIST_RUN_ID = hashlib.sha256(f"{V3_RUN_ID}|{hist_code_hash}|hist_factor_v1".encode()).hexdigest()[:16]
HIST_CKPT_DIR = V3_OUTPUT_ROOT / "checkpoints" / HIST_RUN_ID
HIST_CKPT_DIR.mkdir(parents=True, exist_ok=True)
hist_predictions = []
for seed in V3_TRAINING_SEEDS:
    checkpoint_path = HIST_CKPT_DIR / f"hist_factor_s{seed}.pt"
    if checkpoint_path.exists():
        payload = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
        assert payload["run_id"] == HIST_RUN_ID
        from industry_gnn.hist_factor import HISTFactorRegressor
        hist_model = HISTFactorRegressor(len(CONFIG["features"]["raw"]), V3_HIDDEN_DIM).to(V3_DEVICE)
        hist_model.load_state_dict(payload["state_dict"])
        hist_history = payload["history"]
        print("RESUME: hist_factor", seed)
    else:
        hist_model, hist_history = train_hist_factor(
            HIST_SAMPLES, len(CONFIG["features"]["raw"]), int(seed), V3_HIDDEN_DIM,
            V3_EPOCHS, V3_PATIENCE, float(CONFIG["models"]["learning_rate"]),
            int(CONFIG["models"]["batch_graphs"]), V3_DEVICE,
        )
        temporary = checkpoint_path.with_suffix(".pt.part")
        torch.save({"run_id": HIST_RUN_ID, "seed": int(seed),
                    "state_dict": {k:v.detach().cpu() for k,v in hist_model.state_dict().items()},
                    "history": hist_history}, temporary)
        temporary.replace(checkpoint_path)
    hist_predictions.append(predict_hist_factor(hist_model, HIST_SAMPLES, int(seed), V3_DEVICE))
    V3_HISTORY.extend([{"model":"hist_factor","seed":seed,**row} for row in hist_history])

HIST_BY_SEED = pd.concat(hist_predictions, ignore_index=True)
V3_ALL_SEED_PREDICTIONS = pd.concat([V3_ALL_SEED_PREDICTIONS, HIST_BY_SEED], ignore_index=True)
PRED_HIST = V3_BRIDGE.v3_pred_to_original(V3_ALL_SEED_PREDICTIONS, "hist_factor", V3_PANEL)
V3_PRED_LIST.append(PRED_HIST)
print("HIST-Factor parameters:", sum(p.numel() for p in hist_model.parameters()))


## 14. 统一严格样本外预测、Top-10 保守回测与统计检验


In [ ]:
from industry_gnn.pipeline import combine_prediction_stage
from industry_gnn.backtest import run_execution_backtest
from industry_gnn.evaluation import (
    investment_metrics, prediction_metrics, paired_rankic_tests, industry_neutral_metrics,
    execution_constraint_audit,
)

REQUIRED_MODELS = list(CONFIG["v4_extension"]["model_set"])
assert int(CONFIG["v4_extension"]["primary_portfolio_top_k"]) == 10
selected_frames = [frame for frame in V3_PRED_LIST if frame["model"].iloc[0] in REQUIRED_MODELS]
assert sorted(frame["model"].iloc[0] for frame in selected_frames) == sorted(REQUIRED_MODELS)
PREDICTIONS = combine_prediction_stage(PANEL, selected_frames, PROJECT_ROOT)

top_k = 10
scenario_name = CONFIG["evaluation"]["base_cost_scenario"]
scenario = CONFIG["evaluation"]["cost_scenarios"][scenario_name]
return_frames, trade_frames = [], []
for model_name, model_predictions in PREDICTIONS.groupby("model"):
    returns, trades = run_execution_backtest(
        model_predictions, EXECUTION_PRICES, BENCHMARK_PRICES, top_k,
        scenario_name, scenario, CONFIG["evaluation"]["stamp_duty_schedule"],
    )
    return_frames.append(returns); trade_frames.append(trades)
PORTFOLIO_RETURNS = pd.concat(return_frames, ignore_index=True)
TRADE_LOGS = pd.concat(trade_frames, ignore_index=True)
assert PORTFOLIO_RETURNS.groupby("model").first()["benchmark_return"].eq(0.0).all()

INVESTMENT = investment_metrics(PORTFOLIO_RETURNS)
PREDICTION_METRICS, DAILY_IC = prediction_metrics(
    PREDICTIONS, int(CONFIG["evaluation"]["bootstrap_repetitions"]),
    int(CONFIG["evaluation"]["bootstrap_block_length"]), int(CONFIG["project"]["seed"]),
)
METRICS = INVESTMENT.merge(PREDICTION_METRICS, on="model", how="left")
comparisons = [
    {"name":"Residual-GAT Sparse minus Complete","model_a":"res_gat_sparse","model_b":"res_gat_complete","primary":True},
    {"name":"HIST-Factor minus Residual-GAT Sparse","model_a":"hist_factor","model_b":"res_gat_sparse","primary":True},
    {"name":"Residual-GAT Complete minus Industry MLP","model_a":"res_gat_complete","model_b":"industry_mlp","primary":True},
    {"name":"Residual-GCN minus Industry MLP","model_a":"res_gcn_complete","model_b":"industry_mlp","primary":False},
]
PAIRED_TESTS = paired_rankic_tests(
    DAILY_IC, comparisons, int(CONFIG["evaluation"]["bootstrap_repetitions"]),
    int(CONFIG["evaluation"]["bootstrap_block_length"]), int(CONFIG["project"]["seed"]) + 4000,
)
NEUTRAL = industry_neutral_metrics(
    PREDICTIONS, float(CONFIG["evaluation"]["neutral_prediction_atol"]),
    float(CONFIG["evaluation"]["neutral_prediction_rtol"]),
)
EXECUTION_AUDIT = execution_constraint_audit(TRADE_LOGS)

# Per-seed prediction diagnostics are kept separate from the seed-averaged
# formal predictions. Ridge is deterministic and therefore has no neural seed.
TRAINING_SEED_RESULTS = (
    V3_CORE.rankic_by_date(V3_ALL_SEED_PREDICTIONS, V3_PANEL, period="development")
    .groupby(["model", "seed"], as_index=False)
    .agg(rank_ic_mean=("rank_ic", "mean"), rank_ic_std=("rank_ic", "std"),
         neutral_rank_ic_mean=("neutral_rank_ic", "mean"), dates=("signal_date", "nunique"))
)

# A compact, inspectable ablation table joins paired RankIC inference with
# differences under exactly the same repaired Top-10 backtest.
metric_by_model = METRICS.set_index("model")
ablation_rows = []
for comparison in comparisons:
    model_a, model_b = comparison["model_a"], comparison["model_b"]
    if model_a not in metric_by_model.index or model_b not in metric_by_model.index:
        continue
    paired = PAIRED_TESTS.loc[
        (PAIRED_TESTS["model_a"] == model_a) & (PAIRED_TESTS["model_b"] == model_b)
    ]
    row = paired.iloc[0].to_dict() if len(paired) else {
        "comparison": comparison["name"], "model_a": model_a, "model_b": model_b
    }
    row.update({
        "delta_cumulative_net_return": float(metric_by_model.loc[model_a, "cumulative_net_return"] - metric_by_model.loc[model_b, "cumulative_net_return"]),
        "delta_sharpe_ratio": float(metric_by_model.loc[model_a, "sharpe_ratio"] - metric_by_model.loc[model_b, "sharpe_ratio"]),
        "delta_maximum_drawdown": float(metric_by_model.loc[model_a, "maximum_drawdown"] - metric_by_model.loc[model_b, "maximum_drawdown"]),
        "delta_total_transaction_cost": float(metric_by_model.loc[model_a, "total_transaction_cost"] - metric_by_model.loc[model_b, "total_transaction_cost"]),
        "delta_total_turnover": float(metric_by_model.loc[model_a, "total_turnover"] - metric_by_model.loc[model_b, "total_turnover"]),
    })
    ablation_rows.append(row)
ABLATION_RESULTS = pd.DataFrame(ablation_rows)
TRAINING_HISTORY = pd.DataFrame(V3_HISTORY)
(PROJECT_ROOT / "training_logs").mkdir(parents=True, exist_ok=True)

METRICS.to_csv(PROJECT_ROOT / "metrics.csv", index=False)
PORTFOLIO_RETURNS.to_csv(PROJECT_ROOT / "portfolio_returns.csv", index=False)
TRADE_LOGS.to_csv(PROJECT_ROOT / "trade_logs.csv", index=False)
PAIRED_TESTS.to_csv(PROJECT_ROOT / "paired_rankic_tests.csv", index=False)
NEUTRAL.to_csv(PROJECT_ROOT / "industry_neutral_metrics.csv", index=False)
DAILY_IC.to_csv(PROJECT_ROOT / "tables" / "daily_ic.csv", index=False)
EXECUTION_AUDIT.to_csv(PROJECT_ROOT / "tables" / "execution_constraint_audit.csv", index=False)
ABLATION_RESULTS.to_csv(PROJECT_ROOT / "ablation_results.csv", index=False)
TRAINING_SEED_RESULTS.to_csv(PROJECT_ROOT / "training_seed_results.csv", index=False)
TRAINING_HISTORY.to_csv(PROJECT_ROOT / "training_logs" / "v4_training_history.csv", index=False)
V3_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
METRICS.to_csv(V3_OUTPUT_ROOT / "top10_investment_metrics.csv", index=False)

display(METRICS[[
    "model","top_k","cumulative_net_return","annualized_return","sharpe_ratio",
    "maximum_drawdown","total_transaction_cost","average_daily_turnover","total_turnover",
    "rank_ic_mean","rank_ic_bootstrap_95_low","rank_ic_bootstrap_95_high",
]].sort_values("sharpe_ratio", ascending=False))
display(PAIRED_TESTS)
display(ABLATION_RESULTS)
display(TRAINING_SEED_RESULTS)
display(EXECUTION_AUDIT)


## 15. 图表、研究摘要与结果 ZIP


In [ ]:
import matplotlib.pyplot as plt
from industry_gnn.reporting import create_results_bundle

figure_dir = PROJECT_ROOT / "figures"; figure_dir.mkdir(exist_ok=True)
fig, ax = plt.subplots(figsize=(9, 5))
for model_name, section in PORTFOLIO_RETURNS.groupby("model"):
    section = section.sort_values("date")
    ax.plot(section["date"], (1 + section["net_return"].fillna(0)).cumprod(), label=model_name)
benchmark = PORTFOLIO_RETURNS[PORTFOLIO_RETURNS["model"] == REQUIRED_MODELS[0]].sort_values("date")
ax.plot(benchmark["date"], (1 + benchmark["benchmark_return"].fillna(0)).cumprod(),
        color="black", linestyle="--", label="CSI 300")
ax.set(title="Top-10 portfolio growth - conservative opening execution, base costs",
       xlabel="Date", ylabel="Growth of 1.0")
ax.legend(ncol=2, fontsize=8); fig.tight_layout()
fig.savefig(figure_dir / "v4_top10_conservative_backtest.png", dpi=220, bbox_inches="tight")
plt.show()

best = METRICS.sort_values("sharpe_ratio", ascending=False).iloc[0]
pit_counts = PIT_SUMMARY["status_counts"]
status = "FULL_RUN completed" if RUN_MODE == "FULL_RUN" else "SMOKE_TEST completed; FULL_RUN not run"
summary = f'''# Research Summary - PIT/HIST V4

Status: **{status}**

## Executed scope

- Models: {', '.join(REQUIRED_MODELS)}
- Test interval: {CONFIG['splits']['test_start']} through {CONFIG['splits']['test_end']}
- Portfolio: Top-10, base historical transaction-cost scenario, conservative opening-limit execution
- Highest observed Sharpe in this executed run: `{best['model']}` ({best['sharpe_ratio']:.3f})

## Point-in-Time conclusion

- Audit counts: {pit_counts}
- Strict PIT claim allowed: {PIT_SUMMARY['strict_point_in_time_claim_allowed']}
- Code timing failures stop execution. Missing vendor announcement/revision timestamps remain PARTIAL/UNVERIFIED and are not promoted to VERIFIED.

## HIST interpretation

- `hist_factor` implements predefined industry concepts, hidden concepts, individual residual information and doubly residual forecasts.
- It is a factor-panel adaptation, not an exact reproduction of the paper's Alpha360 GRU and industry/business concept dataset.

## Required files

- `metrics.csv`, `portfolio_returns.csv`, `trade_logs.csv`, `predictions.parquet`
- `paired_rankic_tests.csv`, `ablation_results.csv`, `training_seed_results.csv`, `industry_neutral_metrics.csv`
- `tables/point_in_time_audit.csv`, `tables/point_in_time_audit.json`
- `tables/execution_constraint_audit.csv`, `figures/v4_top10_conservative_backtest.png`
'''
(PROJECT_ROOT / "research_summary.md").write_text(summary, encoding="utf-8")
BUNDLE = create_results_bundle(PROJECT_ROOT)
print("DOWNLOAD:", BUNDLE)


## 16. 结果解释边界

- `PIT_AUDIT` 中任何 `PARTIAL/UNVERIFIED` 都必须随结果保留；它们不会因模型表现好而自动消失。
- 交易执行修复影响收益、Sharpe、回撤、成本和换手，不会自动推翻已有 RankIC；行业/因子时点风险则可能影响模型输入和 RankIC。
- 若 Smoke Test 完成，先检查 `tables/point_in_time_audit.csv` 和交易约束测试，再切换 Full Run。
- Full Run 完成后下载 `research_results_bundle.zip`，把它带回 ChatGPT 做真实结果分析。
